In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes evaluate rouge-score bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 109.3 MB/s eta 0:00:00


In [3]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from kaggle_secrets import UserSecretsClient

# ── Config ──
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
model_id = "meta-llama/Llama-3.1-8B-Instruct"
output_dir = "/kaggle/working/llama31-8b-turkish-sft-v3"
print("V3")

V3


In [4]:
import pandas as pd
from tqdm.auto import tqdm
from peft import PeftModel

In [5]:
# ── Dataset ──
data_files = {
    "train": "/kaggle/input/datasets/berkhanguzeller/capstone-dataset-for-trainingvaltest/sft_train.jsonl",
    "validation": "/kaggle/input/datasets/berkhanguzeller/capstone-dataset-for-trainingvaltest/sft_val.jsonl",
    "test": "/kaggle/input/datasets/berkhanguzeller/capstone-dataset-for-trainingvaltest/sft_test.jsonl",
}
dataset = load_dataset("json", data_files=data_files)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [6]:
SYSTEM_MSG = """You summarize Turkish e-commerce reviews for one target aspect only.

Rules:
- Output language: Turkish.
- Write exactly one short sentence.
- Use only the provided evidence.
- Stay strictly inside the target aspect meaning.
- Do not make numerical or quantity-based claims.
- Do not use words like "çoğu", "birçok", "bazı", "kimi", "genelde", "çoğunlukla" or similar frequency wording.
- Do not mention both positive and negative evidence unless the evidence is strongly and clearly balanced.
- Ignore minor opposing evidence.
- Do not copy review text verbatim.
- If evidence is weak, missing, off-topic, or insufficient, output exactly: null
- Return only the answer text. No JSON. No explanation.
"""

def is_valid_row(example):
    inp = example.get("input_text")
    tgt = example.get("target_text")
    return (
        inp is not None and str(inp).strip() != "" and
        tgt is not None and str(tgt).strip() != ""
    )

def to_prompt_completion(example):
    raw = example["input_text"]

    marker = "\n\nProduct ID:"
    idx = raw.find(marker)
    user_content = raw[idx:].strip() if idx != -1 else raw.strip()

    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_MSG},
            {"role": "user", "content": user_content},
        ],
        "completion": [
            {"role": "assistant", "content": str(example["target_text"]).strip()},
        ],
    }

dataset = dataset.map(
    to_prompt_completion,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/6149 [00:00<?, ? examples/s]

Map:   0%|          | 0/765 [00:00<?, ? examples/s]

Map:   0%|          | 0/774 [00:00<?, ? examples/s]

In [7]:
# ── Model + Quantization ──
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    token=HF_TOKEN,
    device_map="auto",
    quantization_config=bnb_config,
    attn_implementation="eager",
    dtype=torch.float16,
)

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [8]:
# ── LoRA (smaller r = faster) ──
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

In [9]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir=output_dir,
    max_length=672,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    weight_decay=0.01,
    bf16=False,
    fp16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    dataloader_num_workers=2,
    ddp_find_unused_parameters=False,
    completion_only_loss=True,
)

In [10]:
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=lora_config,
    processing_class=tokenizer,
)

Tokenizing train dataset:   0%|          | 0/6149 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/765 [00:00<?, ? examples/s]

In [11]:
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

In [12]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss,Validation Loss
200,0.721162,nan
400,0.635475,nan
600,0.641416,nan
770,0.519607,nan


/usr/local/lib/python3.12/dist-packages/peft/utils/other.py:1394: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-69cbaa6d-1be161ba68f4c63424d26642;f2072eb4-5cad-41a9-9422-4703b3470456)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.1-8B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in. - silently ignoring the lookup for the file config.json in meta-llama/Llama-3.1-8B-Instruct.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:295: UserWarning: Could not find a config file in meta-llama/Llama-3.1-8B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/other.py:1394: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root

TrainOutput(global_step=770, training_loss=0.7470138648887733, metrics={'train_runtime': 27171.7453, 'train_samples_per_second': 0.453, 'train_steps_per_second': 0.028, 'total_flos': 3.274569112212357e+17, 'train_loss': 0.7470138648887733})

In [13]:
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

/usr/local/lib/python3.12/dist-packages/peft/utils/other.py:1394: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-69cbf91e-63dc206c6f1b9aa2651774c5;6c24fb63-39b9-4341-9e15-4e4318f0c704)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.1-8B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in. - silently ignoring the lookup for the file config.json in meta-llama/Llama-3.1-8B-Instruct.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:295: UserWarning: Could not find a config file in meta-llama/Llama-3.1-8B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(


('/kaggle/working/llama31-8b-turkish-sft-v3/tokenizer_config.json',
 '/kaggle/working/llama31-8b-turkish-sft-v3/chat_template.jinja',
 '/kaggle/working/llama31-8b-turkish-sft-v3/tokenizer.json')

# Inference / Testing

In [14]:
# base_model_id = "meta-llama/Llama-3.1-8B-Instruct"
# adapter_path = "/kaggle/input/models/berkhanguzeller/capstone-llama-v1/transformers/default/1/llama31-8b-turkish-sft"

# # =========================
# # Quantization
# # =========================
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,
# )

# # =========================
# # Tokenizer
# # Tek tokenizer yeterli
# # =========================
# tokenizer = AutoTokenizer.from_pretrained(base_model_id, token=HF_TOKEN)

# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = "right"

# # =========================
# # Load NON-TRAINED base model
# # =========================
# base_model_non_trained = AutoModelForCausalLM.from_pretrained(
#     base_model_id,
#     token=HF_TOKEN,
#     device_map="auto",
#     quantization_config=bnb_config,
#     dtype=torch.float16,
# )
# base_model_non_trained.eval()

# # =========================
# # Load TRAINED model (base + LoRA)
# # Ayrı base load ediyoruz ki biri adapter'sız,
# # biri adapter'lı kalsın
# # =========================
# base_model_for_lora = AutoModelForCausalLM.from_pretrained(
#     base_model_id,
#     token=HF_TOKEN,
#     device_map="auto",
#     quantization_config=bnb_config,
#     dtype=torch.float16,
# )

# trained_model = PeftModel.from_pretrained(base_model_for_lora, adapter_path)
# trained_model.eval()

In [15]:
# # =========================
# # Generation helper
# # =========================
# def build_user_content(example):
#     raw = example["input_text"]
#     if raw.startswith("You summarize Turkish"):
#         marker = "\n\nProduct ID:"
#         idx = raw.find(marker)
#         if idx != -1:
#             return raw[idx:].strip()
#     return raw
    
# def generate_one(model, example, max_new_tokens=64):
#     user_content = build_user_content(example)

#     messages = [
#         {"role": "system", "content": SYSTEM_MSG},
#         {"role": "user", "content": user_content},
#     ]

#     enc = tokenizer.apply_chat_template(
#         messages,
#         tokenize=True,
#         add_generation_prompt=True,
#         return_tensors="pt",
#         return_dict=True,
#     )

#     enc = {k: v.to(model.device) for k, v in enc.items()}

#     with torch.no_grad():
#         outputs = model.generate(
#             input_ids=enc["input_ids"],
#             attention_mask=enc["attention_mask"],
#             max_new_tokens=max_new_tokens,
#             do_sample=False,
#             pad_token_id=tokenizer.pad_token_id,
#             eos_token_id=tokenizer.eos_token_id,
#         )

#     pred = tokenizer.decode(
#         outputs[0][enc["input_ids"].shape[-1]:],
#         skip_special_tokens=True
#     ).strip()

#     return pred

In [16]:
# # =========================
# # Inference on validation
# # =========================
# val_ds = dataset["validation"]

# sample_n = 20
# rows = []

# for ex in tqdm(val_ds.select(range(sample_n))):
#     pred_trained = generate_one(trained_model, ex)
#     pred_non_trained = generate_one(base_model_non_trained, ex)

#     rows.append({
#         "input_text": ex["input_text"],
#         "target_text": ex["target_text"],
#         "prediction_trained": pred_trained,
#         "prediction_non_trained": pred_non_trained,
#     })

# df_preds = pd.DataFrame(rows)

In [17]:
# pd.set_option("display.max_colwidth", None)
# df_preds_compare = df_preds[["target_text", "prediction_trained", "prediction_non_trained"]].copy()
# df_preds_compare.head()

In [18]:
# import re

# def extract_user_message_x(text):
#     # \n karakterlerini gerçek newline'a çevir
#     text = text.replace('\\n', '\n')
    
#     # "Product ID:" den itibaren al
#     match = re.search(r'(Product ID:.*)', text, re.DOTALL)
#     if match:
#         return match.group(1).strip()
#     return text.strip()

# df_preds_u = pd.DataFrame()
# df_preds_u['user_message'] = df_preds['input_text'].apply(extract_user_message_x)

In [19]:
# print(df_preds_u['user_message'].iloc[3])

In [20]:
# import pandas as pd
# import numpy as np
# import evaluate
# from tqdm.auto import tqdm

# # =========================
# # 1) Run on FULL validation or a subset
# # =========================
# val_ds = dataset["validation"]

# # quick test:
# #eval_ds = val_ds.select(range(50))

# # full val:
# eval_ds = val_ds

# rows = []
# for ex in tqdm(eval_ds):
#     pred_trained = generate_one(trained_model, ex)
#     pred_non_trained = generate_one(base_model_non_trained, ex)

#     rows.append({
#         "target_text": ex["target_text"],
#         "prediction_trained": pred_trained,
#         "prediction_non_trained": pred_non_trained,
#     })

# df_preds = pd.DataFrame(rows)
# df_preds.head()

In [21]:
# df_preds.to_csv("/kaggle/working/df_preds.csv", index=False)

In [22]:
# df_preds = pd.read_csv("/kaggle/working/df_preds.csv")

In [23]:
# import pandas as pd
# import numpy as np
# import evaluate
# from tqdm.auto import tqdm

In [24]:
# # =========================
# # 2) Load metrics
# # =========================
# rouge = evaluate.load("rouge")
# bertscore = evaluate.load("bertscore")

# # references
# refs = df_preds["target_text"].astype(str).tolist()

# # predictions
# preds_trained = df_preds["prediction_trained"].astype(str).tolist()
# preds_non_trained = df_preds["prediction_non_trained"].astype(str).tolist()

# # =========================
# # 3) ROUGE
# # =========================
# rouge_trained = rouge.compute(
#     predictions=preds_trained,
#     references=refs,
#     use_stemmer=False
# )

# rouge_non_trained = rouge.compute(
#     predictions=preds_non_trained,
#     references=refs,
#     use_stemmer=False
# )

# print("TRAINED ROUGE")
# print(rouge_trained)

# print("\nNON-TRAINED ROUGE")
# print(rouge_non_trained)

In [25]:
# !pip install -q -U bert-score

In [26]:
# from transformers import BertTokenizer

# # Monkey-patch the missing method
# if not hasattr(BertTokenizer, 'build_inputs_with_special_tokens'):
#     def _build_inputs_with_special_tokens(self, token_ids_0, token_ids_1=None):
#         cls = [self.cls_token_id]
#         sep = [self.sep_token_id]
#         if token_ids_1 is None:
#             return cls + token_ids_0 + sep
#         return cls + token_ids_0 + sep + token_ids_1 + sep
#     BertTokenizer.build_inputs_with_special_tokens = _build_inputs_with_special_tokens

# preds_trained = [str(x).strip() if str(x).strip() else " " for x in preds_trained]
# preds_non_trained = [str(x).strip() if str(x).strip() else " " for x in preds_non_trained]
# refs = [str(x).strip() if str(x).strip() else " " for x in refs]

# from bert_score import score as bert_score_fn

# # Trained
# P_t, R_t, F1_t = bert_score_fn(preds_trained, refs, lang="tr", verbose=True, model_type="bert-base-multilingual-cased",)
# print("TRAINED BERTScore")
# print({
#     "precision": P_t.mean().item(),
#     "recall": R_t.mean().item(),
#     "f1": F1_t.mean().item(),
# })

# # Non-trained
# P_nt, R_nt, F1_nt = bert_score_fn(preds_non_trained, refs, lang="tr", verbose=True, model_type="bert-base-multilingual-cased",)
# print("\nNON-TRAINED BERTScore")
# print({
#     "precision": P_nt.mean().item(),
#     "recall": R_nt.mean().item(),
#     "f1": F1_nt.mean().item(),
# })

In [27]:
# results = pd.DataFrame([
#     {
#         "model": "trained",
#         "rouge1": rouge_trained["rouge1"],
#         "rouge2": rouge_trained["rouge2"],
#         "rougeL": rouge_trained["rougeL"],
#         "rougeLsum": rouge_trained["rougeLsum"],
#         "bertscore_precision": P_t.mean().item(),
#         "bertscore_recall": R_t.mean().item(),
#         "bertscore_f1": F1_t.mean().item(),
#     },
#     {
#         "model": "non_trained",
#         "rouge1": rouge_non_trained["rouge1"],
#         "rouge2": rouge_non_trained["rouge2"],
#         "rougeL": rouge_non_trained["rougeL"],
#         "rougeLsum": rouge_non_trained["rougeLsum"],
#         "bertscore_precision": P_nt.mean().item(),
#         "bertscore_recall": R_nt.mean().item(),
#         "bertscore_f1": F1_nt.mean().item(),
#     }
# ])

# results